# Telugu Akshara-wise Metrics for 12 Models — Fully Corrected

This Kaggle notebook evaluates the predictions already present in the supplied Excel files. It does **not** retrain the models.

It creates one Excel workbook with exactly three sheets: **Train**, **Test**, and **Val**. Each sheet has one row per Akshara for each of the 12 model combinations:

- 2-State: BiLSTM, CNN, GRU, RNN
- 4-State: BiLSTM, CNN, GRU, RNN
- 4-State + CRF: BiLSTM, CNN, GRU, RNN

Output columns: `Char`, `Freq`, `Complexity`, `Accuracy`, `Precision`, `Recall`, `F1`, `Model Name`, `State`.

**Important label-format fix:** the prediction workbooks store label sequences as continuous strings (for example, `001010` or `BEBIES`), not as whitespace-separated tokens. This notebook converts each string into individual labels before aligning it with Unicode grapheme clusters in `INPUT_SENTENCE`. This prevents empty confusion matrices and all-zero metrics caused by parsing an entire sequence as one label.

`Complexity` stores the seven-slot vector `[Base, Pollu, Vottu 1, Vottu 2, Vottu 3, Maatra, Sunna]`, not a summed score. Example vectors: `ఆ -> [1, 0, 0, 0, 0, 0, 0]`; `కా -> [1, 0, 0, 0, 0, 1, 0]`; `ర్తి -> [1, 0, 1, 0, 0, 1, 0]`.

Precision, Recall, and F1 are macro-averaged across the label classes for each character. A character with no evaluated occurrences in a split will correctly have zero metrics for that split/model.


In [27]:
!pip -q install regex openpyxl xlsxwriter

In [28]:
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import regex

warnings.filterwarnings("ignore")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.3.3
NumPy: 2.1.3


## 1. Configure input files

Attach the Kaggle dataset containing the four Excel files. The paths below use the dataset directory provided in the question. If your Kaggle mount path differs, set the corresponding value in `MANUAL_PATHS`.

The file finder accepts either an exact full path or searches Kaggle input directories using only the filename (not the full absolute path).


In [29]:
FREQUENCY_FILE = "/kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_akshara_frequencies.xlsx"

PREDICTION_FILES = {
    "2-State": "/kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_2_state_model_predictions_detailed_results.xlsx",
    "4-State": "/kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_4_state_model_predictions_detailed_results.xlsx",
    "4-State + CRF": "/kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_4_state_crf_model_predictions_detailed_results.xlsx",
}

MANUAL_PATHS = {
    "frequency": None,
    "2-State": None,
    "4-State": None,
    "4-State + CRF": None,
}

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "telugu_akshara_metrics_12_models_with_vector_complexity.xlsx"


def find_input_file(filepath, manual_path=None):
    if manual_path:
        p = Path(manual_path)
        if p.is_file():
            return p
        raise FileNotFoundError(f"Manual path does not exist: {manual_path}")

    # First try the exact path supplied.
    p = Path(filepath)
    if p.is_file():
        return p

    # Kaggle paths may be mounted under a slightly different root.
    # rglob must receive a filename pattern, not an absolute path.
    filename = p.name
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        f"Could not find {filename}. Confirm the Kaggle dataset is attached "
        "or set its full path in MANUAL_PATHS."
    )


FILE_PATHS = {
    "frequency": find_input_file(FREQUENCY_FILE, MANUAL_PATHS["frequency"]),
    **{
        state: find_input_file(filepath, MANUAL_PATHS[state])
        for state, filepath in PREDICTION_FILES.items()
    },
}

for key, value in FILE_PATHS.items():
    print(f"{key}: {value}")
print("Output:", OUTPUT_FILE)


frequency: /kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_akshara_frequencies.xlsx
2-State: /kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_2_state_model_predictions_detailed_results.xlsx
4-State: /kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_4_state_model_predictions_detailed_results.xlsx
4-State + CRF: /kaggle/input/datasets/abhaygokavarapu0630/teluguexcelsafteroutputsentencelevel/telugu_4_state_crf_model_predictions_detailed_results.xlsx
Output: /kaggle/working/telugu_akshara_metrics_12_models_with_vector_complexity.xlsx


## 2. Load the frequency and prediction workbooks

Frequency sheets:
- `Training` → output `Train`
- `Testing` → output `Test`
- `Validation` → output `Val`

Prediction sheets:
- `train` → output `Train`
- `test` → output `Test`
- `Evaluation` → output `Val`


In [30]:
frequency_sheet_map = {
    "Train": "Training",
    "Test": "Testing",
    "Val": "Validation",
}
prediction_sheet_map = {
    "Train": "train",
    "Test": "test",
    "Val": "Evaluation",
}

frequency_data = {
    output_split: pd.read_excel(FILE_PATHS["frequency"], sheet_name=source_sheet)
    for output_split, source_sheet in frequency_sheet_map.items()
}

prediction_data = {}
required_columns = {
    "INPUT_SENTENCE",
    "GROUND_TRUTH_LABELS",
    "PREDICTED_LABELS",
    "MODEL",
}

for state, path in FILE_PATHS.items():
    if state == "frequency":
        continue

    prediction_data[state] = {}
    xls = pd.ExcelFile(path)

    for output_split, source_sheet in prediction_sheet_map.items():
        if source_sheet not in xls.sheet_names:
            raise ValueError(
                f"Missing sheet '{source_sheet}' in {path.name}. "
                f"Available sheets: {xls.sheet_names}"
            )

        df = pd.read_excel(path, sheet_name=source_sheet, dtype=str)
        missing = required_columns - set(df.columns)
        if missing:
            raise ValueError(
                f"{path.name}/{source_sheet} is missing columns: {sorted(missing)}"
            )

        for col in required_columns:
            df[col] = df[col].fillna("").astype(str)

        prediction_data[state][output_split] = df

# Validate frequency sheets and report model counts.
for split, df in frequency_data.items():
    if not {"Character", "Frequency"}.issubset(df.columns):
        raise ValueError(
            f"Frequency sheet for {split} must contain Character and Frequency."
        )
    print(f"{split}: {len(df):,} frequency rows")

for state, split_data in prediction_data.items():
    for split, df in split_data.items():
        models = sorted(df["MODEL"].dropna().unique().tolist())
        print(f"{state} | {split}: {len(df):,} rows | models: {models}")


Train: 6,623 frequency rows
Test: 3,398 frequency rows
Val: 3,355 frequency rows
2-State | Train: 40,544 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
2-State | Test: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
2-State | Val: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State | Train: 40,544 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State | Test: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State | Val: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State + CRF | Train: 40,544 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State + CRF | Test: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']
4-State + CRF | Val: 8,688 rows | models: ['BILSTM', 'CNN', 'GRU', 'RNN']


## 3. Label parsing, seven-slot complexity, and metric functions

The input sentence is split into Unicode grapheme clusters (`\X`) so Telugu combining marks remain attached to their base Akshara.

**Critical:** label strings are parsed character by character. For example, `00101` becomes `['0','0','1','0','1']`, and `BEBIS` becomes `['B','E','B','I','S']`. These are aligned position-by-position with the Aksharas in the input sentence.

For each Akshara, a confusion matrix is accumulated with ground-truth labels as rows and predicted labels as columns. Accuracy is `correct / total occurrences` for that Akshara. Precision, Recall, and F1 are macro averages across the applicable state labels.


In [31]:
LABELS_BY_STATE = {
    "2-State": ["0", "1"],
    "4-State": ["B", "I", "E", "S"],
    "4-State + CRF": ["B", "I", "E", "S"],
}


def split_aksharas(text):
    """Split Telugu text into Unicode grapheme clusters."""
    return regex.findall(r"\X", str(text or ""))


def parse_label_sequence(value, allowed_labels):
    """
    Parse continuous or whitespace-separated label sequences.

    Examples:
      '00101'       -> ['0', '0', '1', '0', '1']
      'BEBIES'      -> ['B', 'E', 'B', 'I', 'E', 'S']
      'B E B I E S' -> ['B', 'E', 'B', 'I', 'E', 'S']
    """
    value = str(value).strip()
    compact = "".join(value.split())
    allowed = set(allowed_labels)

    # The provided prediction files use continuous label strings.
    if compact and all(char in allowed for char in compact):
        return list(compact)

    # Fallback for genuinely token-separated labels.
    tokens = value.split()
    if tokens and all(token in allowed for token in tokens):
        return tokens

    # Return compact characters so invalid symbols are counted explicitly
    # by the confusion-building function rather than silently disappearing.
    return list(compact)


# Seven-slot vector:
# [Base, Pollu/Virama, Vottu 1, Vottu 2, Vottu 3, Maatra, Sunna]
TELUGU_CONSONANTS = set(range(0x0C15, 0x0C3A)) | {0x0C58, 0x0C59, 0x0C5A}
TELUGU_VIRAMA = 0x0C4D
TELUGU_MAATRAS = (
    set(range(0x0C3E, 0x0C4D))
    | {0x0C55, 0x0C56, 0x0C62, 0x0C63}
)
TELUGU_SUNNA = {0x0C02, 0x0C03}


def akshara_complexity(akshara):
    """Return the full seven-dimensional structural vector as Excel text."""
    cps = [ord(ch) for ch in str(akshara)]
    if not cps:
        vector = [0, 0, 0, 0, 0, 0, 0]
        return "[" + ", ".join(map(str, vector)) + "]"

    base = 1
    # Pollu is a virama at the end of this grapheme cluster.
    pollu = int(cps[-1] == TELUGU_VIRAMA)
    maatra = int(any(cp in TELUGU_MAATRAS for cp in cps))
    sunna = int(any(cp in TELUGU_SUNNA for cp in cps))

    # A consonant immediately following virama is a vottu component.
    vottu_count = sum(
        1
        for j in range(len(cps) - 1)
        if cps[j] == TELUGU_VIRAMA and cps[j + 1] in TELUGU_CONSONANTS
    )

    vector = [
        base,
        pollu,
        int(vottu_count >= 1),
        int(vottu_count >= 2),
        int(vottu_count >= 3),
        maatra,
        sunna,
    ]
    return "[" + ", ".join(map(str, vector)) + "]"


def calculate_four_metrics(cm):
    """Calculate accuracy and macro precision, recall, and F1."""
    cm = np.asarray(cm, dtype=np.int64)
    total = int(cm.sum())

    if total == 0:
        return {
            "Accuracy": 0.0,
            "Precision": 0.0,
            "Recall": 0.0,
            "F1": 0.0,
        }

    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0).astype(float) - tp
    fn = cm.sum(axis=1).astype(float) - tp

    accuracy = float(tp.sum() / total)

    precision_per_class = np.divide(
        tp,
        tp + fp,
        out=np.zeros_like(tp, dtype=float),
        where=(tp + fp) != 0,
    )
    recall_per_class = np.divide(
        tp,
        tp + fn,
        out=np.zeros_like(tp, dtype=float),
        where=(tp + fn) != 0,
    )
    f1_per_class = np.divide(
        2 * precision_per_class * recall_per_class,
        precision_per_class + recall_per_class,
        out=np.zeros_like(tp, dtype=float),
        where=(precision_per_class + recall_per_class) != 0,
    )

    return {
        "Accuracy": accuracy,
        "Precision": float(np.mean(precision_per_class)),
        "Recall": float(np.mean(recall_per_class)),
        "F1": float(np.mean(f1_per_class)),
    }


def build_character_confusions(model_df, labels):
    """
    Build one label confusion matrix per Akshara.

    Confusion matrix convention:
      rows = ground-truth label
      columns = predicted label

    Returns:
      character_matrices, mismatch_rows, aligned_positions, invalid_positions
    """
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    character_matrices = {}
    mismatch_rows = 0
    aligned_positions = 0
    invalid_positions = 0

    for row in model_df.itertuples(index=False):
        sentence = str(getattr(row, "INPUT_SENTENCE", "") or "")
        true_value = str(getattr(row, "GROUND_TRUTH_LABELS", "") or "")
        pred_value = str(getattr(row, "PREDICTED_LABELS", "") or "")

        chars = split_aksharas(sentence)
        true_labels = parse_label_sequence(true_value, labels)
        pred_labels = parse_label_sequence(pred_value, labels)

        if not (len(chars) == len(true_labels) == len(pred_labels)):
            mismatch_rows += 1

        common_length = min(len(chars), len(true_labels), len(pred_labels))

        for idx in range(common_length):
            char = chars[idx]
            true_label = true_labels[idx]
            pred_label = pred_labels[idx]

            if true_label not in label_to_idx or pred_label not in label_to_idx:
                invalid_positions += 1
                continue

            if char not in character_matrices:
                character_matrices[char] = np.zeros(
                    (len(labels), len(labels)), dtype=np.int64
                )

            true_idx = label_to_idx[true_label]
            pred_idx = label_to_idx[pred_label]
            character_matrices[char][true_idx, pred_idx] += 1
            aligned_positions += 1

    return (
        character_matrices,
        mismatch_rows,
        aligned_positions,
        invalid_positions,
    )


# Sanity-check the vector examples supplied by the user.
assert akshara_complexity("ఆ") == "[1, 0, 0, 0, 0, 0, 0]"
assert akshara_complexity("కా") == "[1, 0, 0, 0, 0, 1, 0]"
assert akshara_complexity("ర్తి") == "[1, 0, 1, 0, 0, 1, 0]"

print("Metric and complexity functions are ready.")
print("Complexity examples:")
for example in ["ఆ", "కా", "ర్తి"]:
    print(example, "->", akshara_complexity(example))


Metric and complexity functions are ready.
Complexity examples:
ఆ -> [1, 0, 0, 0, 0, 0, 0]
కా -> [1, 0, 0, 0, 0, 1, 0]
ర్తి -> [1, 0, 1, 0, 0, 1, 0]


## 4. Build the Train, Test, and Val result sheets

For every split, the notebook loops over all three segmentation states and all four architectures. Each character's confusion matrix is used to calculate the four requested metrics.

The notebook also records alignment diagnostics. If a character has no aligned occurrences for a model/split, its metrics are zero because there is no evidence to evaluate for that character. The diagnostic output helps distinguish this from a parsing or alignment failure.


In [32]:
result_sheets = {}
alignment_notes = []
state_order = ["2-State", "4-State", "4-State + CRF"]

for split in ["Train", "Test", "Val"]:
    freq_df = frequency_data[split].copy()
    freq_df["Character"] = freq_df["Character"].fillna("").astype(str)

    # Keep one frequency entry per character, preserving the sheet order.
    freq_df = freq_df[["Character", "Frequency"]].copy()
    freq_df["Frequency"] = pd.to_numeric(
        freq_df["Frequency"], errors="coerce"
    ).fillna(0)

    rows = []

    for state in state_order:
        labels = LABELS_BY_STATE[state]
        pred_df = prediction_data[state][split]

        model_names = sorted(pred_df["MODEL"].dropna().unique().tolist())
        if len(model_names) != 4:
            print(
                f"WARNING: expected 4 architectures for {state}/{split}, "
                f"found {len(model_names)}: {model_names}"
            )

        for model_name in model_names:
            model_df = pred_df[pred_df["MODEL"] == model_name]

            (
                char_matrices,
                mismatch_rows,
                aligned_positions,
                invalid_positions,
            ) = build_character_confusions(model_df, labels)

            alignment_notes.append({
                "Split": split,
                "State": state,
                "Model Name": model_name,
                "Rows": len(model_df),
                "Rows with length mismatch": mismatch_rows,
                "Aligned label positions": aligned_positions,
                "Invalid label positions": invalid_positions,
                "Aksharas with evaluated occurrences": len(char_matrices),
            })

            for char, freq in zip(freq_df["Character"], freq_df["Frequency"]):
                cm = char_matrices.get(
                    char,
                    np.zeros((len(labels), len(labels)), dtype=np.int64),
                )
                metrics = calculate_four_metrics(cm)

                rows.append({
                    "Char": char,
                    "Freq": int(freq),
                    "Complexity": akshara_complexity(char),
                    **metrics,
                    "Model Name": model_name,
                    "State": state,
                })

    result_sheets[split] = pd.DataFrame(rows, columns=[
        "Char",
        "Freq",
        "Complexity",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Model Name",
        "State",
    ])

    combinations = result_sheets[split][
        ["Model Name", "State"]
    ].drop_duplicates().shape[0]

    nonzero_rows = int((result_sheets[split]["Accuracy"] > 0).sum())

    print(
        f"{split}: {len(result_sheets[split]):,} output rows | "
        f"{combinations} model/state combinations | "
        f"{nonzero_rows:,} rows with non-zero accuracy"
    )

alignment_df = pd.DataFrame(alignment_notes)

# Stop rather than silently export an empty/invalid evaluation.
if alignment_df.empty or alignment_df["Aligned label positions"].sum() == 0:
    raise ValueError(
        "No valid aligned label positions were evaluated. "
        "Check sentence/label formats and the input sheet mapping."
    )

if (alignment_df["Invalid label positions"] > 0).any():
    print("WARNING: Some positions contained labels outside the expected label set.")
    display(alignment_df[alignment_df["Invalid label positions"] > 0])

print("\nAlignment diagnostics:")
display(alignment_df)


Train: 79,476 output rows | 12 model/state combinations | 61,079 rows with non-zero accuracy
Test: 40,776 output rows | 12 model/state combinations | 30,804 rows with non-zero accuracy
Val: 40,260 output rows | 12 model/state combinations | 30,472 rows with non-zero accuracy

Alignment diagnostics:


,Split,State,Model Name,Rows,Rows with length mismatch,Aligned label positions,Invalid label positions,Aksharas with evaluated occurrences
0,Train,2-State,BILSTM,10136,5,379681,0,6624
1,Train,2-State,CNN,10136,5,379681,0,6624
2,Train,2-State,GRU,10136,5,379681,0,6624
3,Train,2-State,RNN,10136,5,379681,0,6624
4,Train,4-State,BILSTM,10136,5,379681,0,6624
5,Train,4-State,CNN,10136,5,379681,0,6624
6,Train,4-State,GRU,10136,5,379681,0,6624
7,Train,4-State,RNN,10136,5,379681,0,6624
8,Train,4-State + CRF,BILSTM,10136,5,379681,0,6624
9,Train,4-State + CRF,CNN,10136,5,379681,0,6624


## 5. Export the Excel workbook

The output workbook has exactly three sheets: `Train`, `Test`, and `Val`. Each sheet has the same nine columns and includes all available model/state combinations.


In [33]:
with pd.ExcelWriter(OUTPUT_FILE, engine="xlsxwriter") as writer:
    workbook = writer.book

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#244062",
        "text_wrap": True,
        "valign": "vcenter",
    })
    metric_fmt = workbook.add_format({"num_format": "0.0000"})

    for split, df in result_sheets.items():
        df.to_excel(writer, sheet_name=split, index=False)
        ws = writer.sheets[split]

        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)
        ws.set_row(0, 26, header_fmt)

        ws.set_column("A:A", 14)
        ws.set_column("B:B", 12)
        ws.set_column("C:C", 28)
        ws.set_column("D:G", 14, metric_fmt)
        ws.set_column("H:H", 18)
        ws.set_column("I:I", 18)

print("Created:", OUTPUT_FILE)
print("Size (MB):", round(OUTPUT_FILE.stat().st_size / (1024 * 1024), 2))


Created: /kaggle/working/telugu_akshara_metrics_12_models_with_vector_complexity.xlsx
Size (MB): 5.84


## 6. Verify the workbook

This cell checks that the Excel file contains exactly the three requested sheets, the expected columns, and the model/state combinations. It also displays a preview and alignment diagnostics.


In [34]:
check = pd.ExcelFile(OUTPUT_FILE)
print("Sheets:", check.sheet_names)

expected_sheets = ["Train", "Test", "Val"]
expected_columns = [
    "Char", "Freq", "Complexity", "Accuracy", "Precision",
    "Recall", "F1", "Model Name", "State"
]

if check.sheet_names != expected_sheets:
    raise AssertionError(
        f"Expected sheets {expected_sheets}, found {check.sheet_names}"
    )

for split in expected_sheets:
    df = pd.read_excel(OUTPUT_FILE, sheet_name=split)

    print(f"\n{split}: {len(df):,} rows")
    print("Columns correct:", df.columns.tolist() == expected_columns)
    print(
        "Model/state combinations:",
        df[["Model Name", "State"]].drop_duplicates().shape[0],
    )
    print("Non-zero accuracy rows:", int((df["Accuracy"] > 0).sum()))
    display(df.head())

print("\nAlignment diagnostics:")
display(alignment_df)

print("\nExcel output is ready:", OUTPUT_FILE)
print("In Kaggle, download it from the notebook Output panel.")


Sheets: ['Train', 'Test', 'Val']

Train: 79,476 rows
Columns correct: True
Model/state combinations: 12
Non-zero accuracy rows: 61079


,Char,Freq,Complexity,Accuracy,Precision,Recall,F1,Model Name,State
0,ని,7827,"[1, 0, 0, 0, 0, 1, 0]",0.771432,0.748407,0.737480,0.742091,BILSTM,2-State
1,అ,7477,"[1, 0, 0, 0, 0, 0, 0]",0.838973,0.786182,0.733269,0.753372,BILSTM,2-State
2,ది,7229,"[1, 0, 0, 0, 0, 1, 0]",0.821690,0.819086,0.819751,0.819401,BILSTM,2-State
3,న,7124,"[1, 0, 0, 0, 0, 0, 0]",0.794778,0.790410,0.787412,0.788723,BILSTM,2-State
4,రి,6913,"[1, 0, 0, 0, 0, 1, 0]",0.812816,0.727945,0.612061,0.631817,BILSTM,2-State



Test: 40,776 rows
Columns correct: True
Model/state combinations: 12
Non-zero accuracy rows: 30804


,Char,Freq,Complexity,Accuracy,Precision,Recall,F1,Model Name,State
0,అ,1681,"[1, 0, 0, 0, 0, 0, 0]",0.834622,0.778693,0.724707,0.744879,BILSTM,2-State
1,ని,1578,"[1, 0, 0, 0, 0, 1, 0]",0.768695,0.749725,0.741105,0.744784,BILSTM,2-State
2,రి,1532,"[1, 0, 0, 0, 0, 1, 0]",0.810052,0.723222,0.608526,0.627152,BILSTM,2-State
3,ది,1526,"[1, 0, 0, 0, 0, 1, 0]",0.810616,0.810981,0.809662,0.810027,BILSTM,2-State
4,న,1478,"[1, 0, 0, 0, 0, 0, 0]",0.788227,0.779721,0.776120,0.777744,BILSTM,2-State



Val: 40,260 rows
Columns correct: True
Model/state combinations: 12
Non-zero accuracy rows: 30472


,Char,Freq,Complexity,Accuracy,Precision,Recall,F1,Model Name,State
0,అ,1643,"[1, 0, 0, 0, 0, 0, 0]",0.826537,0.781930,0.696912,0.722501,BILSTM,2-State
1,ని,1585,"[1, 0, 0, 0, 0, 1, 0]",0.761514,0.739432,0.737471,0.738415,BILSTM,2-State
2,ది,1539,"[1, 0, 0, 0, 0, 1, 0]",0.816764,0.816486,0.813052,0.814308,BILSTM,2-State
3,న,1489,"[1, 0, 0, 0, 0, 0, 0]",0.777703,0.772569,0.773117,0.772834,BILSTM,2-State
4,రి,1487,"[1, 0, 0, 0, 0, 1, 0]",0.813046,0.704101,0.596413,0.613082,BILSTM,2-State



Alignment diagnostics:


,Split,State,Model Name,Rows,Rows with length mismatch,Aligned label positions,Invalid label positions,Aksharas with evaluated occurrences
0,Train,2-State,BILSTM,10136,5,379681,0,6624
1,Train,2-State,CNN,10136,5,379681,0,6624
2,Train,2-State,GRU,10136,5,379681,0,6624
3,Train,2-State,RNN,10136,5,379681,0,6624
4,Train,4-State,BILSTM,10136,5,379681,0,6624
5,Train,4-State,CNN,10136,5,379681,0,6624
6,Train,4-State,GRU,10136,5,379681,0,6624
7,Train,4-State,RNN,10136,5,379681,0,6624
8,Train,4-State + CRF,BILSTM,10136,5,379681,0,6624
9,Train,4-State + CRF,CNN,10136,5,379681,0,6624



Excel output is ready: /kaggle/working/telugu_akshara_metrics_12_models_with_vector_complexity.xlsx
In Kaggle, download it from the notebook Output panel.
